In [39]:

import pandas as pd
from Lmd_Utils import cleantext
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from sklearn.metrics.pairwise import cosine_similarity
simu=0.8

In [40]:
questions=[]
answers=[]

In [41]:
df=pd.read_parquet('./Data/train.parquet')
df2=pd.read_parquet('./Data/validation.parquet')
df = pd.concat([df, df2], ignore_index=True)

In [42]:

df.columns

Index(['id', 'uit_id', 'title', 'context', 'question', 'answers',
       'is_impossible', 'plausible_answers'],
      dtype='str')

In [43]:
df.head()

,id,uit_id,title,context,question,answers,is_impossible,plausible_answers
0,0001-0001-0001,uit_000001,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...,"{'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}",False,None
1,0001-0001-0002,uit_000002,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà ...,"{'text': ['Thủ tướng'], 'answer_start': [60]}",False,None
2,0001-0001-0003,uit_000003,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,"Giai đoạn năm 1955-1976, Phạm Văn Đồng nắm giữ...",{'text': ['Thủ tướng Chính phủ Việt Nam Dân ch...,False,None
3,0001-0001-0004,uit_000004,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng trước k...,"{'text': [], 'answer_start': []}",True,"{'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}"
4,0001-0001-0005,uit_000005,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Hồ Học Lãm giữ chức vụ gì trong bộ máy Nhà nướ...,"{'text': [], 'answer_start': []}",True,"{'text': ['Thủ tướng'], 'answer_start': [60]}"


In [44]:
df.iloc[0]

id                                                      0001-0001-0001
uit_id                                                      uit_000001
title                                                    Phạm Văn Đồng
context              Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...
question             Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...
answers               {'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}
is_impossible                                                    False
plausible_answers                                                 None
Name: 0, dtype: object

In [45]:
df['question'].iloc[3]


'Tên gọi nào được Phạm Văn Đồng sử dụng trước khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm?'

In [46]:
df['context'].iloc[3]

'Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).'

In [47]:
df['answers'].iloc[3]

{'text': array([], dtype=object), 'answer_start': array([], dtype=int32)}

In [48]:
df['plausible_answers'].iloc[3]

{'text': array(['Lâm Bá Kiệt'], dtype=object),
 'answer_start': array([507], dtype=int32)}

In [49]:
for i in range(len(df)):
    is_imp = df['is_impossible'].iloc[i]
    ans_dict = df['answers'].iloc[i]
    plausible_dict = df['plausible_answers'].iloc[i]
    ques_text = df['question'].iloc[i]

    if is_imp == True:
        if isinstance(plausible_dict, dict) and len(plausible_dict.get('text', [])) > 0:
            answers.append(cleantext(plausible_dict['text'][0]))
            questions.append(cleantext(ques_text))
    else:
        if isinstance(ans_dict, dict) and len(ans_dict.get('text', [])) > 0:
            answers.append(cleantext(ans_dict['text'][0]))
            questions.append(cleantext(ques_text))

print(len(questions))
print(len(answers))
print(len(df))

32268
32268
32268


In [50]:
vect=TfidfVectorizer()
X_train=vect.fit_transform(questions)
joblib.dump(vect,'./models_TF-IDF/tfidf_QA_VN.pkl')
joblib.dump(X_train,'./models_TF-IDF/X_train_QA_VN.pkl')
joblib.dump(answers,'./models_TF-IDF/Answer_QA_VN.pkl')
joblib.dump(questions,'./models_TF-IDF/Question_QA_VN.pkl')


['./models_TF-IDF/Question_QA_VN.pkl']